# Aula 05 (MLCB) - AC-2 Parte 1
Exercícios 1 a 4: Lematização, Embeddings, Regressão Logística e KNN

## Exercício 1 - Normalização Textual e Lematização

In [1]:
# ------------------------------------------------------------------------------
# EXERCÍCIO 1: Pipeline de Normalização Textual e Lematização
# ------------------------------------------------------------------------------
import re
import nltk
from nltk.corpus import stopwords
import spacy

nltk.download('stopwords', quiet=True)
palavras_vazias_pt = set(stopwords.words('portuguese'))

# Carregar o pipeline linguístico do Spacy para o Português
nlp = spacy.load("pt_core_news_sm")

def normalizar_e_lematizar(texto):
    """
    Trata uma string de texto cru em três passos:
    1. Deixa tudo em caixa baixa
    2. Descarta símbolos, sinais de pontuação e dígitos com Regex
    3. Elimina palavras vazias e reduz cada termo ao seu lema com o Spacy
    """
    # Passar para caixa baixa
    texto_tratado = texto.lower()

    # Manter apenas letras e espaços em branco (re.sub descarta o resto)
    texto_tratado = re.sub(r'[^a-záàâãéèêíïóôõöúçñ\s]', '', texto_tratado)

    # Rodar o pipeline do Spacy sobre o texto já tratado
    doc = nlp(texto_tratado)

    # Coletar o lema (token.lemma_) descartando palavras vazias e espaços
    lemas_selecionados = [
        token.lemma_ for token in doc
        if token.text not in palavras_vazias_pt and not token.is_space and len(token.text) > 1
    ]

    return " ".join(lemas_selecionados)

# === TESTE DO EXERCÍCIO 1 ===
frase_exemplo = "Gostaria de saber se vocês estão DEVOLVENDO os valores das mesas compradas!!!"
print("--- RESULTADOS DO EXERCÍCIO 1 ---")
print("Frase Original:", frase_exemplo)
print("Frase Normalizada & Lematizada:", normalizar_e_lematizar(frase_exemplo))

--- RESULTADOS DO EXERCÍCIO 1 ---
Frase Original: Gostaria de saber se vocês estão DEVOLVENDO os valores das mesas compradas!!!
Frase Normalizada & Lematizada: gostar saber devolver valor meso comprada


## Exercício 2 - Representação de Sentenças (Média de Vetores)

In [2]:
# ------------------------------------------------------------------------------
# EXERCÍCIO 2: Representação de Sentenças via Embeddings e Média de Vetores
# ------------------------------------------------------------------------------
import numpy as np
import pandas as pd
import gensim.downloader as api

print("\n--- RESULTADOS DO EXERCÍCIO 2 ---")
print("Baixando modelo de Embeddings (Gensim)...")
# Modelo pré-treinado disponibilizado pelo Gensim (versão enxuta de 50 dimensões para uso em aula)
modelo_embeddings = api.load("glove-wiki-gigaword-50")

def vetorizar_frase_media(frase, model):
    """
    Transforma uma frase inteira em um único vetor denso tirando a
    média (Mean Pooling) dos vetores de cada palavra conhecida.
    """
    termos = frase.split()
    lista_vetores = []

    for termo in termos:
        if termo in model:
            # Guardar o vetor do termo na lista acumuladora
            lista_vetores.append(model[termo])

    if len(lista_vetores) == 0:
        # Nenhum termo presente no vocabulário -> devolve vetor nulo
        return np.zeros(model.vector_size)

    # Média elemento a elemento ao longo do eixo 0
    vetor_resultante = np.mean(lista_vetores, axis=0)
    return vetor_resultante

# === TESTE DO EXERCÍCIO 2 ===
df = pd.read_csv("sac_moveis_ac2.csv")
matriz_vetores = np.array([vetorizar_frase_media(msg, modelo_embeddings) for msg in df['mensagem']])
print("Formato da Matriz de Vetores Densos (Exemplos, Dimensões):", matriz_vetores.shape)


--- RESULTADOS DO EXERCÍCIO 2 ---
Baixando modelo de Embeddings (Gensim)...
Formato da Matriz de Vetores Densos (Exemplos, Dimensões): (32, 50)


## Exercício 3 - Regressão Logística com Transbordo

In [3]:
# ------------------------------------------------------------------------------
# EXERCÍCIO 3: Regressão Logística para Intenções com Regra de Transbordo
# ------------------------------------------------------------------------------
from sklearn.linear_model import LogisticRegression

print("\n--- RESULTADOS DO EXERCÍCIO 3 ---")

# 1. Rótulos (a matriz de vetores já foi construída no Exercício 2)
y = df['intencao']

# Ajustar o classificador de Regressão Logística
modelo_logistico = LogisticRegression(max_iter=1000)
modelo_logistico.fit(matriz_vetores, y)

def classificar_intencao_transbordo(mensagem_cliente, modelo, model_emb, limiar=0.50):
    """
    Prevê a intenção da mensagem e confere se a probabilidade máxima
    alcança o nível mínimo de confiança exigido.
    """
    # 1. Converter a mensagem do cliente em vetor denso
    vetor_mensagem = vetorizar_frase_media(mensagem_cliente, model_emb).reshape(1, -1)

    # Probabilidade estimada para cada classe (predict_proba)
    vetor_probabilidades = modelo.predict_proba(vetor_mensagem)[0]

    # Maior probabilidade e a classe correspondente
    prob_maxima = np.max(vetor_probabilidades)
    posicao_classe = np.argmax(vetor_probabilidades)
    intencao_estimada = modelo.classes_[posicao_classe]

    # Regra de transbordo (fallback humano)
    if prob_maxima < limiar:
        return "FALLBACK_HUMANO", prob_maxima
    else:
        return intencao_estimada, prob_maxima

# === TESTE DO EXERCÍCIO 3 ===
casos_teste = [
    "Quero saber o valor do frete do sofá",             # Intenção esperada: vendas_orcamento
    "Gostaria de ver receitas de bolo de cenoura"       # Frase fora do domínio -> deve cair no transbordo
]

for frase in casos_teste:
    intencao, conf = classificar_intencao_transbordo(frase, modelo_logistico, modelo_embeddings)
    print(f"Frase: '{frase}' | Resultado: {intencao} | Confiança: {conf:.2%}")


--- RESULTADOS DO EXERCÍCIO 3 ---
Frase: 'Quero saber o valor do frete do sofá' | Resultado: FALLBACK_HUMANO | Confiança: 35.52%
Frase: 'Gostaria de ver receitas de bolo de cenoura' | Resultado: vendas_orcamento | Confiança: 69.10%


## Exercício 4 - Comparação com KNN

In [4]:
# ------------------------------------------------------------------------------
# EXERCÍCIO 4: KNN no Espaço Denso frente à Regressão Logística
# ------------------------------------------------------------------------------
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

print("\n--- RESULTADOS DO EXERCÍCIO 4 ---")

# Criar e ajustar o KNN usando n_neighbors=3
modelo_vizinhos = KNeighborsClassifier(n_neighbors=3)
modelo_vizinhos.fit(matriz_vetores, y)

# Previsões das duas estratégias sobre a mesma base
pred_logistico = modelo_logistico.predict(matriz_vetores)
pred_vizinhos = modelo_vizinhos.predict(matriz_vetores)

# Acurácia de cada abordagem
acuracia_logistico = accuracy_score(y, pred_logistico)
acuracia_vizinhos = accuracy_score(y, pred_vizinhos)

print(f"Acurácia - Regressão Logística (Linear): {acuracia_logistico:.2%}")
print(f"Acurácia - KNN (Distância K=3): {acuracia_vizinhos:.2%}")

# Pergunta para reflexão do aluno:
# Qual das duas abordagens se sai melhor com frases muito curtas ou afastadas no espaço vetorial?


--- RESULTADOS DO EXERCÍCIO 4 ---
Acurácia - Regressão Logística (Linear): 93.75%
Acurácia - KNN (Distância K=3): 56.25%
